<img src="https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/images/edrai_logo.png" alt="EDR|AI" width="300"/>

# Chapter 41 — Natural Experiments: Difference-in-Differences, Discontinuities, and Interrupted Time Series

This is the **companion notebook** of [Chapter 41 — Natural Experiments: Difference-in-Differences, Discontinuities, and Interrupted Time Series](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part7-further-routes/natural-experiments.html) from **EDR|AI — Evidence-Driven Research in the Age of AI**. Authored by [Davi Moreira](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/index.html).

[Open the chapter](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part7-further-routes/natural-experiments.html) · [Book home](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/index.html) · [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html)

*AI is your arm and your research assistant, not your brain.*

## How to use this notebook

1. Work top to bottom, with the chapter open in another tab.
2. Copy each **AI prompt** into your AI tool, run it, then record in the response cell what came back and what you verified.
3. Run the code cells; change something and run again.
4. Finish the **It is your turn** workspace at the end — that is this chapter's step of your own research project.
5. Log every AI use in your **AI Research Ledger**: task · tool · prompt · output summary · decision · verification method · remaining concern · you as the responsible researcher.
6. Your AI can be more than a chatbot: agentic tools can run multi-step work for you. Delegating boldly is fine; reviewing, curating, and deciding stay yours.

> **The research decision.** When a law, a date, or a cutoff assigned the treatment
> instead of you, decide which comparison you are willing to defend as the world that
> did not happen, and write down the one assumption that makes it stand in. The rule
> is in the documents and the assumption is in your sentence, so no tool gets to
> declare that the comparison it found is the counterfactual.

## Code from the chapter

The cells below come from the chapter. Run them, then change something and run again — the numbers should move the way the chapter says they will.

*From the section “A worked example”.* **What this cell does:** exactly what the chapter walks through in that section; run it and compare with the chapter.

In [ ]:
import numpy as np, pandas as pd
SEED = 464
rng = np.random.default_rng(SEED)

n = 200                                   # restaurants on each side of the border
# Constructed world: a regional downturn cuts 2.0 workers EVERYWHERE,
# and the wage raise itself cuts 1.0 more, only in State A.
state = np.repeat(["A (raise)", "B (no raise)"], n)
before = rng.normal(20, 4, size=2 * n)
after = before - 2.0 - 1.0 * (state == "A (raise)") + rng.normal(0, 1.5, size=2 * n)
df = pd.DataFrame({"state": state, "change": after - before})

g = df.groupby("state")["change"].agg(["mean", "var", "size"])
a, b = g.loc["A (raise)"], g.loc["B (no raise)"]
did = a["mean"] - b["mean"]
se = np.sqrt(a["var"] / a["size"] + b["var"] / b["size"])
print(f"true effect of the raise           : {-1.0:+.2f} workers")
print(f"before/after, State A only         : {a['mean']:+.2f}")
print(f"before/after, State B only         : {b['mean']:+.2f}")
print(f"difference-in-differences          : {did:+.2f}  "
      f"(95% interval {did - 1.96*se:+.2f} to {did + 1.96*se:+.2f})")
print("\nthe before/after number blames the raise for the downturn;")
print("State B is what lets you subtract it, IF A would have moved like B")

**Reading the output.** The chapter reads this output in the same section; check yours against it, then change one input and rerun. The numbers should move the way the chapter says they will.

*From the section “A seeded simulation”.* **What this cell does:** exactly what the chapter walks through in that section; run it and compare with the chapter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 464
rng = np.random.default_rng(SEED)

# Event study: 200 stores per state, quarters -4..3, raise at quarter 0.
n, quarters, tau = 200, np.arange(-4, 4), -1.0
def event_study(extra_trend):             # extra_trend < 0: A was ALREADY sliding
    base = rng.normal(20, 4, size=(2 * n, 1))
    shock = rng.normal(0, 0.5, size=quarters.size)       # hits both states
    treated = np.repeat([True, False], n)[:, None]
    y = (base + shock + extra_trend * quarters * treated
         + tau * treated * (quarters >= 0)
         + rng.normal(0, 1.5, size=(2 * n, quarters.size)))
    d = y - y[:, [3]]                                     # change since quarter -1
    est = d[:n].mean(0) - d[n:].mean(0)
    se = np.sqrt(d[:n].var(0, ddof=1) / n + d[n:].var(0, ddof=1) / n)
    change = y[:, 4:].mean(1) - y[:, :4].mean(1)          # after avg minus before avg
    did = change[:n].mean() - change[n:].mean()
    did_se = np.sqrt(change[:n].var(ddof=1) / n + change[n:].var(ddof=1) / n)
    return est, se, did, did_se
par, par_se, did_par, se_par = event_study(0.0)
vio, vio_se, did_vio, se_vio = event_study(-0.4)
for name, est, did, s in (("parallel", par, did_par, se_par),
                          ("sliding ", vio, did_vio, se_vio)):
    print(f"{name}: leads {np.round(est[:3], 2)}, after {np.round(est[4:], 2)}, "
          f"overall {did:+.2f} ({did - 1.96*s:+.2f} to {did + 1.96*s:+.2f})")

# Regression discontinuity: grant at score >= 70, true jump 1.5 jobs.
m, cut, jump = 2000, 70, 1.5
score = rng.uniform(40, 100, m)
x = score - cut
noise = rng.normal(0, 2.0, m)
jobs = 5 + 0.08 * x + 0.0008 * x**2 + jump * (x >= 0) + noise
# Stress test: the same applicants and noise, but an S-shaped curve that climbs
# fastest near the cutoff and levels off at both ends. The true jump is still 1.5.
bent = 5 + 3 * np.tanh(x / 12) + jump * (x >= 0) + noise
def rd(h, y=jobs):                         # local straight lines within h points
    ends, ses = [], []
    for side in (x < 0, x >= 0):
        k = side & (np.abs(x) <= h)
        X = np.column_stack([np.ones(k.sum()), x[k]])
        b = np.linalg.lstsq(X, y[k], rcond=None)[0]
        r = y[k] - X @ b
        ses.append(np.sqrt((r @ r / (k.sum() - 2)) * np.linalg.inv(X.T @ X)[0, 0]))
        ends.append(b)
    return ends[1][0] - ends[0][0], np.hypot(*ses), ends
for h in (5, 10, 20):
    e, s, _ = rd(h)
    print(f"bandwidth {h:2d}: jump {e:.2f} (95% interval {e-1.96*s:.2f} to {e+1.96*s:.2f})")
for h in (5, 10, 20):
    e, s, _ = rd(h, bent)
    print(f"S-curve, bandwidth {h:2d}: jump {e:.2f} ({e-1.96*s:.2f} to {e+1.96*s:.2f})")
print("applicants scoring 65-69.9:", ((x >= -5) & (x < 0)).sum(),
      "| scoring 70-74.9:", ((x >= 0) & (x < 5)).sum())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 3.8))
ax1.errorbar(quarters - .12, par, yerr=1.96 * par_se, fmt="o", color="#2a78d6",
             capsize=2, label="parallel trends hold")
ax1.errorbar(quarters + .12, vio, yerr=1.96 * vio_se, fmt="s", color="#eb6834",
             capsize=2, label="treated state already sliding")
ax1.plot([-0.5, 3.4], [tau, tau], color="#333333", ls="--")
ax1.axhline(0, color="#bbbbbb"); ax1.legend(fontsize=8)
edges = np.arange(40, 100.01, 2.5); bins = np.digitize(score, edges) - 1
ax2.scatter((edges[:-1] + edges[1:]) / 2,
            [jobs[bins == i].mean() for i in range(edges.size - 1)], color="#777777")
_, _, (b0, b1) = rd(10)                    # the two local lines at bandwidth 10
xl, xr = np.linspace(-10, 0, 20), np.linspace(0, 10, 20)
ax2.plot(cut + xl, b0[0] + b0[1] * xl, color="#2a78d6")
ax2.plot(cut + xr, b1[0] + b1[1] * xr, color="#2a78d6")
ax2.axvline(cut, color="#333333", ls="--")
plt.show()

**Reading the output.** The chapter reads this output in the same section; check yours against it, then change one input and rerun. The numbers should move the way the chapter says they will.

## It is your turn

<!-- station-pointer:begin -->
> **A further route beyond the five pathways.** This lesson
> extends [Studio 5: Develop the pathway](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/studios/studio05-develop-the-pathway.html). Read it once
> you have declared your primary pathway and your question
> calls for this design. Studio 5's milestone asks for the
> same decisions, answered for this route.
<!-- station-pointer:end -->

*You have a causal question and a rule that assigned treatment without asking you.
This further route is where that rule becomes a comparison you can defend in
writing, or an honest statement that it cannot be one yet.*

The hands-on half of this section lives in the chapter's **companion notebook**: open it in Colab with the badge at the top, and work the steps there.

Commit your own answer first, then delegate. Each prompt below is a checkable job,
not a request for a verdict. Work them as a loop. The first answer is a draft: find
the claim you cannot check, say so in your next message, and run it again. Some
tools will run that whole loop unattended and hand you a finished event study. The
finished look is exactly what makes the questions in this chapter worth asking
before you accept it.

> **Do not delegate.**
>
> Three calls stay yours. You decide **what the assignment rule actually was**, read
> from the primary document, including when it was announced and who could see it
> coming. You decide **which comparison stands in for the world without the rule**,
> and you write the assumption that lets it. And you decide **whether the checks leave
> that assumption plausible enough to carry *because***, or whether your answer is
> causal, currently unidentified. A tool can fit the model, draw the plot, and find
> papers. The counterfactual you defend carries your name.

**Step 1.** Quote the assignment rule from its primary document. Write the exact sentence of
the law, manual, or policy that decided treatment, with its source, its
effective date, and its announcement date. Then write which of the three designs
it fits: a date with an untreated comparison group (difference-in-differences), a
cutoff on a score (regression discontinuity), or one population treated at once
(interrupted time series).

*When you are ready to delegate this step:*

```text
Act as a policy research librarian. I am studying [the policy] in [place] around
[date]. Find the primary documents that set the rule: the statute or regulation,
the agency notice, and any published scoring or eligibility sheet. For each, give
the issuing body, the document title, the date, and where I can retrieve it. Only
include documents you are confident exist; mark anything uncertain.
```

After running, verify: open every document yourself and find the sentence that
assigns treatment; cut any item you cannot retrieve. Counters **confident
fabrication** (an invented statute number arrives as confidently as a real one).

✍️ **Your work for step 1.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 2.** Write your Research Contract lines for this route. Your question, as a treatment
and an outcome. Your estimand, in one sentence that names whose effect, over which
periods, or at which cutoff. Your data strategy: the units, the periods or score
range, and the comparison group. Your answer strategy: the difference you will
compute and the interval you will report, including how that interval accounts
for the policy changing place by place rather than store by store. Your warrant:
parallel trends, continuity at the cutoff, or a stable pre-trend, written in
plain words a skeptic could argue with.

✍️ **Your work for step 2.** Double-click this cell and write your answer here.

**Step 3.** Name the threat most likely to break your warrant, and the check you will run.
For difference-in-differences, an event study with every lead shown, and a note on
whether units adopted on different dates. For a discontinuity, the density check,
covariate balance at the cutoff, and the estimate at two or three bandwidths you
fix before looking, with a bias-aware interval and a second curve shape. For an interrupted
time series, seasonality and the competing events of that same date, plus a
comparison series.

*When you are ready to delegate this step:*

```text
Act as a hostile labor economist. Here is my design: [paste rule, comparison,
outcome, periods]. List every event, trend, or behavior that could move my
treated group differently from my comparison group around [date], in a table with
the threat, the direction it would push my estimate, and the data I could use to
check it. Then name the single threat you think I am most likely to miss.
```

After running, verify: compare the table with the threat you wrote first, and for
each row find one real source that says the event happened when and where the
tool says. Counters **illusion of completeness** (a tidy table that omits the one
competing event that shares your date).

✍️ **Your work for step 3.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 4.** Run a falsification test and say in advance what it would look like if your
design were in trouble. Move the policy date to an earlier year when nothing
happened, using only data from before the real policy. Or move the cutoff to a
score where nothing changes, using only applicants on one side of the real cutoff
so its jump cannot leak in. Or use an outcome the policy should not touch. A placebo estimate far from zero is a warning. One near zero
lowers your concern about that specific threat, as far as the test was sensitive
to it, and leaves the rest of the warrant where it was.

*When you are ready to delegate this step:*

```text
Act as a skeptical referee. Here is my design and the choices I fixed BEFORE
seeing the estimate: [paste rule, comparison, outcome, periods or bandwidth].
Propose three placebo tests for it: a fake policy date, a fake cutoff, or an
outcome the policy should not touch. For each, tell me what result would worry
me and roughly how large an effect the test could detect with my sample. Do not
suggest changing my bandwidth, comparison group, or window to improve my main
result.
```

After running, verify: open the documents again and confirm that no real rule
changed at each fake date or cutoff, then run every placebo yourself and check
that any number the tool quotes appears in your own output. If it slips in a new
bandwidth or window that makes the main result significant, reject it and note it
in your ledger. Counters **sycophantic agreement** (a helper that finds the result
you seemed to want).

✍️ **Your work for step 4.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 5.** Write the boundary sentence. State what your estimate describes, for whom, and
over which periods or at which cutoff, with its interval, and name the one
assumption a reader must accept for the word *because* to stand. If your checks
left the assumption implausible, write the causal-language boundary instead: what
the comparison shows, and the *because* you are declining to claim.

✍️ **Your work for step 5.** Double-click this cell and write your answer here.

**Step 6.** Log the step in your AI Research Ledger, and verify at least one output with a
named method from the [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html). A
**falsification test** is the natural check for this route: the placebo date,
placebo cutoff, or untouched outcome from step 4. **Simulation** is the strong
second check: build a small world in which your comparison group was already
drifting, as the orange world above was, and see whether your event study would
have shown it. An AI reviewer may run the checks with you; the decision to accept
or reject stays yours.

✍️ **Your work for step 6.** Double-click this cell and write your answer here.

### The standard this section is held to

Use this as a self-check while you work. It is also the bar the same work meets later, once your project carries it. Each row: **0** missing, **1** attempted but incomplete, generic, or unverified, **2** complete, specific to your own project, and verified where a check applies. **14 points in all.**

| # | Criterion | 0–2 |
|---|---|---|
| Step 1 | Quote the assignment rule from its primary document | |
| Step 2 | Write your Research Contract lines for this route | |
| Step 3 | Name the threat most likely to break your warrant, and the check you will run | |
| Step 4 | Run a falsification test and say in advance what it would look like if your design were in trouble | |
| Step 5 | Write the boundary sentence | |
| Step 6 | Log the step in your AI Research Ledger, and verify at least one output with a named method from the Verification Guide | |
| + | Craft and verification record: AI use logged in your AI Research Ledger, claims stated with their uncertainty, and each key claim verified with a named method | |

In [ ]:
# Scratch space — use this cell for any code your steps need.

**Before you leave this notebook:** add today's rows to your AI Research Ledger, and verify your key claim with a named method from the [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html). AI can review AI — but the last decision is human.

Next: [Chapter 42 — Survey Experiments: Vignettes, Conjoint, and List Experiments](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part7-further-routes/survey-experiments.html). That chapter may not be on your route — [Studio 5: Develop the pathway](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/studios/studio05-develop-the-pathway.html) is the junction; follow the lesson that matches your own project.